# 🤖 Paper MM Dashboard — BTC Up/Down Market Maker
Live paper trading simulation with real Polymarket orderbooks.

In [ ]:
# Install dependencies
import subprocess
result = subprocess.run(
    ["pip", "install", "httpx", "rich", "ipywidgets", "-q"],
    capture_output=True, text=True
)
print("✅ Dependencies installed")

In [ ]:
import sys, os

# Add repo root to path
REPO_ROOT = "/content/ouroboros_repo"
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Check the bot is importable
try:
    from bots.polymarket.paper_mm import PaperMMEngine
    from bots.polymarket.strategy_btc_mm import BTCUpDownMM, is_updown_market, parse_window_from_question
    from bots.polymarket.scanner import discover_markets, get_orderbook
    from bots.polymarket.fair_price import bootstrap_from_klines, get_btc_state, fetch_btc_price, compute_updown_fair_price
    print("✅ Bot modules loaded")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Make sure the repo is at /content/ouroboros_repo")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
GAMMA          = 0.5    # Risk aversion (higher = wider spreads, faster inventory unwind)
ORDER_SIZE     = 5.0    # USDC per fill
REQUOTE_SEC    = 10.0   # Requote interval (seconds)
MAX_MARKETS    = 3      # Max simultaneous markets
REPORT_SEC     = 30.0   # Dashboard refresh interval
DRIVE_ROOT     = "/content/drive/MyDrive/Ouroboros"

print(f"Config: γ={GAMMA} | size=${ORDER_SIZE} | requote={REQUOTE_SEC}s | max_markets={MAX_MARKETS}")

In [ ]:
import time
from datetime import datetime, timezone
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

def fmt_pnl(val: float) -> str:
    """Format PnL with color."""
    sign = "+" if val >= 0 else ""
    color = "#00c851" if val >= 0 else "#ff4444"
    return f'<span style="color:{color}; font-weight:bold">{sign}${val:.4f}</span>'

def fmt_side_bar(yes_fills: int, no_fills: int) -> str:
    """Visual bar showing YES/NO fill distribution."""
    total = yes_fills + no_fills
    if total == 0:
        return "⬜⬜⬜⬜⬜ (no fills)"
    yes_pct = yes_fills / total
    yes_blocks = int(yes_pct * 10)
    no_blocks = 10 - yes_blocks
    bar = "🟢" * yes_blocks + "🔴" * no_blocks
    return f"{bar} YES:{yes_fills} NO:{no_fills}"

def fmt_inventory_bar(net_yes: float, net_no: float) -> str:
    """Visual bar for inventory balance."""
    total = abs(net_yes) + abs(net_no)
    if total < 0.01:
        return "⚖️ Balanced"
    yes_pct = abs(net_yes) / total if net_yes > 0 else 0
    no_pct = abs(net_no) / total if net_no > 0 else 0
    if net_yes > net_no:
        return f"📈 Long YES: {net_yes:.2f} | NO: {net_no:.2f}"
    elif net_no > net_yes:
        return f"📉 Long NO: {net_no:.2f} | YES: {net_yes:.2f}"
    else:
        return f"⚖️ YES: {net_yes:.2f} | NO: {net_no:.2f}"

def render_dashboard(engine, btc_state, elapsed_sec: float, cycle: int):
    """Render the complete dashboard as HTML."""
    states = list(engine.states.values())
    total_pnl = engine.get_total_pnl()
    total_quotes = sum(s.quote_count for s in states)
    total_fills = sum(len(s.fills) for s in states)
    fill_rate = total_fills / total_quotes * 100 if total_quotes else 0

    open_states = [s for s in states if not s.resolved]
    resolved_states = [s for s in states if s.resolved]

    elapsed_str = f"{int(elapsed_sec//60)}m {int(elapsed_sec%60)}s"
    btc_price = btc_state.prices[-1] if btc_state.prices else 0
    sigma = btc_state.sigma_annualized()

    # Header
    pnl_color = "#00c851" if total_pnl >= 0 else "#ff4444"
    pnl_sign = "+" if total_pnl >= 0 else ""

    html_parts = [f"""
    <div style="font-family: monospace; background: #1e1e1e; color: #d4d4d4; padding: 16px; border-radius: 8px; max-width: 800px;">

    <div style="display:flex; justify-content:space-between; align-items:center; border-bottom: 1px solid #444; padding-bottom:8px; margin-bottom:12px;">
      <div style="font-size:18px; font-weight:bold; color:#61dafb;">🤖 Paper MM Dashboard</div>
      <div style="color:#888; font-size:12px;">cycle #{cycle} | running {elapsed_str} | {datetime.now().strftime('%H:%M:%S')}</div>
    </div>

    <div style="display:grid; grid-template-columns:repeat(4, 1fr); gap:12px; margin-bottom:16px;">
      <div style="background:#2d2d2d; padding:10px; border-radius:6px; text-align:center;">
        <div style="color:#888; font-size:11px;">TOTAL P&L</div>
        <div style="font-size:22px; color:{pnl_color}; font-weight:bold;">{pnl_sign}${total_pnl:.4f}</div>
      </div>
      <div style="background:#2d2d2d; padding:10px; border-radius:6px; text-align:center;">
        <div style="color:#888; font-size:11px;">BTC PRICE</div>
        <div style="font-size:22px; color:#ffd700; font-weight:bold;">${btc_price:,.0f}</div>
      </div>
      <div style="background:#2d2d2d; padding:10px; border-radius:6px; text-align:center;">
        <div style="color:#888; font-size:11px;">VOLATILITY σ</div>
        <div style="font-size:22px; color:#c792ea; font-weight:bold;">{sigma*100:.1f}%</div>
      </div>
      <div style="background:#2d2d2d; padding:10px; border-radius:6px; text-align:center;">
        <div style="color:#888; font-size:11px;">FILL RATE</div>
        <div style="font-size:22px; color:#82aaff; font-weight:bold;">{fill_rate:.1f}%</div>
      </div>
    </div>

    <div style="display:grid; grid-template-columns:repeat(3, 1fr); gap:12px; margin-bottom:16px;">
      <div style="background:#2d2d2d; padding:8px 10px; border-radius:6px;">
        <span style="color:#888; font-size:11px;">Quotes:</span> <span style="color:#fff;">{total_quotes}</span>
      </div>
      <div style="background:#2d2d2d; padding:8px 10px; border-radius:6px;">
        <span style="color:#888; font-size:11px;">Fills:</span> <span style="color:#fff;">{total_fills}</span>
      </div>
      <div style="background:#2d2d2d; padding:8px 10px; border-radius:6px;">
        <span style="color:#888; font-size:11px;">Markets:</span> <span style="color:#fff;">{len(open_states)} open / {len(resolved_states)} resolved</span>
      </div>
    </div>
    """]

    # Open markets table
    if open_states:
        html_parts.append("""
    <div style="margin-bottom:12px;">
      <div style="color:#61dafb; font-size:13px; font-weight:bold; margin-bottom:8px;">📊 Open Positions</div>
      <table style="width:100%; border-collapse:collapse; font-size:12px;">
        <thead>
          <tr style="color:#888; border-bottom:1px solid #444;">
            <th style="text-align:left; padding:4px 6px;">Market</th>
            <th style="text-align:center; padding:4px;">P_fair</th>
            <th style="text-align:center; padding:4px;">Bid/Ask</th>
            <th style="text-align:center; padding:4px;">Fills</th>
            <th style="text-align:center; padding:4px;">Inventory</th>
            <th style="text-align:right; padding:4px;">P&L</th>
          </tr>
        </thead>
        <tbody>
        """)

        for s in open_states:
            yes_fills = sum(1 for f in s.fills if f.side == "YES")
            no_fills = sum(1 for f in s.fills if f.side == "NO")
            inv_str = fmt_inventory_bar(s.net_yes_qty, s.net_no_qty)
            pnl_val = s.realized_pnl
            pnl_c = "#00c851" if pnl_val >= 0 else "#ff4444"
            pnl_s = "+" if pnl_val >= 0 else ""

            # Truncate question
            q = s.market_question
            if len(q) > 45:
                q = q[:42] + "..."

            html_parts.append(f"""
          <tr style="border-bottom:1px solid #333;">
            <td style="padding:5px 6px; color:#e0e0e0;">{q}</td>
            <td style="text-align:center; color:#ffd700; padding:5px;">{s.last_p_fair:.3f}</td>
            <td style="text-align:center; color:#82aaff; padding:5px; font-size:11px;">{s.last_bid:.3f}/{s.last_ask:.3f}</td>
            <td style="text-align:center; padding:5px;">
              <span style="color:#00c851;">Y:{yes_fills}</span> <span style="color:#ff6b6b;">N:{no_fills}</span>
            </td>
            <td style="text-align:center; color:#c792ea; font-size:11px; padding:5px;">{inv_str}</td>
            <td style="text-align:right; color:{pnl_c}; padding:5px 6px;">{pnl_s}${pnl_val:.4f}</td>
          </tr>
            """)

        html_parts.append("</tbody></table></div>")

    # Recent fills
    all_fills = []
    for s in states:
        all_fills.extend(s.fills)
    all_fills.sort(key=lambda f: f.ts, reverse=True)
    recent_fills = all_fills[:8]

    if recent_fills:
        html_parts.append("""
    <div style="margin-bottom:12px;">
      <div style="color:#61dafb; font-size:13px; font-weight:bold; margin-bottom:8px;">⚡ Recent Fills</div>
      <table style="width:100%; border-collapse:collapse; font-size:11px;">
        <thead>
          <tr style="color:#888; border-bottom:1px solid #444;">
            <th style="text-align:left; padding:3px 6px;">Time</th>
            <th style="text-align:center; padding:3px;">Side</th>
            <th style="text-align:center; padding:3px;">Direction</th>
            <th style="text-align:center; padding:3px;">Price</th>
            <th style="text-align:center; padding:3px;">Size</th>
            <th style="text-align:left; padding:3px 6px;">Market</th>
          </tr>
        </thead>
        <tbody>
        """)

        for f in recent_fills:
            ts_str = datetime.fromtimestamp(f.ts).strftime("%H:%M:%S")
            side_color = "#00c851" if f.side == "YES" else "#ff6b6b"
            dir_color = "#82aaff" if f.direction == "BUY" else "#ffd700"
            dir_icon = "🔼" if f.direction == "BUY" else "🔽"
            q_short = f.market_question[:40] + "..." if len(f.market_question) > 40 else f.market_question

            html_parts.append(f"""
          <tr style="border-bottom:1px solid #2a2a2a;">
            <td style="padding:3px 6px; color:#888;">{ts_str}</td>
            <td style="text-align:center; color:{side_color}; padding:3px; font-weight:bold;">{f.side}</td>
            <td style="text-align:center; color:{dir_color}; padding:3px;">{dir_icon} {f.direction}</td>
            <td style="text-align:center; color:#fff; padding:3px;">{f.price:.4f}</td>
            <td style="text-align:center; color:#c792ea; padding:3px;">${f.size_usdc:.2f}</td>
            <td style="padding:3px 6px; color:#aaa;">{q_short}</td>
          </tr>
            """)

        html_parts.append("</tbody></table></div>")

    # Resolved markets
    if resolved_states:
        html_parts.append("""
    <div>
      <div style="color:#61dafb; font-size:13px; font-weight:bold; margin-bottom:8px;">✅ Resolved</div>
      <table style="width:100%; border-collapse:collapse; font-size:12px;">
        <thead>
          <tr style="color:#888; border-bottom:1px solid #444;">
            <th style="text-align:left; padding:3px 6px;">Market</th>
            <th style="text-align:center; padding:3px;">Outcome</th>
            <th style="text-align:center; padding:3px;">Fills</th>
            <th style="text-align:right; padding:3px 6px;">Total P&L</th>
          </tr>
        </thead>
        <tbody>
        """)

        for s in resolved_states[-5:]:
            m_pnl = s.realized_pnl + (s.resolution_pnl or 0.0)
            pnl_c = "#00c851" if m_pnl >= 0 else "#ff4444"
            pnl_s = "+" if m_pnl >= 0 else ""
            out_color = "#00c851" if s.outcome == "UP" else "#ff4444"
            q = s.market_question[:45] + "..." if len(s.market_question) > 45 else s.market_question

            html_parts.append(f"""
          <tr style="border-bottom:1px solid #2a2a2a;">
            <td style="padding:3px 6px; color:#aaa;">{q}</td>
            <td style="text-align:center; color:{out_color}; font-weight:bold; padding:3px;">{s.outcome}</td>
            <td style="text-align:center; color:#888; padding:3px;">{len(s.fills)}</td>
            <td style="text-align:right; color:{pnl_c}; padding:3px 6px;">{pnl_s}${m_pnl:.4f}</td>
          </tr>
            """)

        html_parts.append("</tbody></table></div>")

    html_parts.append("</div>")

    return "".join(html_parts)

print("✅ Dashboard functions loaded")

In [ ]:
import asyncio
import httpx
import time
from IPython.display import display, HTML, clear_output

async def run_paper_mm_dashboard(
    gamma: float = GAMMA,
    order_size_usdc: float = ORDER_SIZE,
    requote_interval: float = REQUOTE_SEC,
    max_markets: int = MAX_MARKETS,
    duration_sec: float = 600.0,  # Run for 10 minutes by default
):
    """
    Paper MM with live in-notebook dashboard.
    Refreshes display every ~5 seconds.
    """
    from bots.polymarket.paper_mm import PaperMMEngine
    from bots.polymarket.strategy_btc_mm import (
        BTCUpDownMM, is_updown_market, parse_window_from_question, BothSidesQuotes
    )
    from bots.polymarket.scanner import discover_markets, get_orderbook
    from bots.polymarket.fair_price import (
        bootstrap_from_klines, get_btc_state, fetch_btc_price
    )

    engine = PaperMMEngine(order_size_usdc=order_size_usdc)
    start_time = time.monotonic()
    cycle = 0
    active_tasks = {}

    def select_markets(all_markets):
        now = time.time()
        tradeable = []
        for m in all_markets:
            if not is_updown_market(m.question):
                continue
            window = parse_window_from_question(m.question)
            if not window:
                continue
            _, end_ts = window
            if end_ts - now < 60:
                continue
            tradeable.append((m, end_ts))
        tradeable.sort(key=lambda x: x[1])
        return [m for m, _ in tradeable[:max_markets]]

    async def run_market(market):
        window = parse_window_from_question(market.question)
        if not window:
            return
        start_ts, end_ts = window
        mm = BTCUpDownMM(
            market=market,
            clob_client=None,
            gamma=gamma,
            order_size_usdc=order_size_usdc,
            requote_interval=requote_interval,
            paper=True,
        )
        while time.time() < end_ts - 5:
            try:
                quotes = mm.compute_quotes()
                ob_yes, ob_no = await asyncio.gather(
                    get_orderbook(client, market.yes_token_id),
                    get_orderbook(client, market.no_token_id),
                )
                fills = engine.process_quote(
                    market, quotes, ob_yes, ob_no,
                    only_reduce=quotes.only_reduce,
                )
                for fill in fills:
                    if fill.direction == "BUY":
                        mm.inventory.update_buy(fill.side, fill.qty, fill.price)
                    else:
                        mm.inventory.update_sell(fill.side, fill.qty, fill.price)
                    mm.fill_count += 1
                mm.quote_count += 1
            except Exception as e:
                pass  # Silently skip errors in background tasks
            remaining = max(0, end_ts - time.time())
            sleep = min(requote_interval, remaining)
            if sleep <= 0:
                break
            await asyncio.sleep(sleep)
        # Resolve
        state = engine.states.get(market.id)
        if state and not state.resolved:
            btc_state = get_btc_state()
            recent_ret = btc_state.momentum_return(lookback_seconds=300)
            outcome = "UP" if recent_ret >= 0 else "DOWN"
            engine.resolve_market(market.id, outcome)

    # Bootstrap price data
    print("🔄 Bootstrapping BTC price data from Binance...")
    bootstrap_from_klines()
    btc_state = get_btc_state()
    print(f"✅ BTC price data ready: {len(btc_state.prices)} ticks, σ={btc_state.sigma_annualized()*100:.1f}%/yr")
    print(f"\n▶️ Starting Paper MM (duration: {duration_sec:.0f}s)...\n")

    async with httpx.AsyncClient() as client:
        # Initial market discovery
        try:
            all_markets = await discover_markets(client)
            selected = select_markets(all_markets)
            print(f"📌 Found {len(selected)} tradeable markets")
            for m in selected:
                print(f"  • {m.question[:70]}")
        except Exception as e:
            print(f"⚠️ Market discovery error: {e}")
            selected = []

        # Launch market tasks
        for market in selected:
            if market.id not in active_tasks:
                task = asyncio.create_task(run_market(market))
                active_tasks[market.id] = task

        print("\n" + "="*60)
        print("Dashboard refreshes every 5 seconds. Stop with ⏹ button.\n")

        last_discovery = time.monotonic()
        last_refresh = 0

        try:
            while time.monotonic() - start_time < duration_sec:
                now = time.monotonic()
                elapsed = now - start_time
                cycle += 1

                # Refresh BTC state
                try:
                    fetch_btc_price()
                    btc_state = get_btc_state()
                except Exception:
                    pass

                # Re-discover markets every 60s
                if now - last_discovery >= 60:
                    try:
                        all_markets = await discover_markets(client)
                        new_markets = select_markets(all_markets)
                        for market in new_markets:
                            if market.id not in active_tasks and market.id not in engine.states:
                                task = asyncio.create_task(run_market(market))
                                active_tasks[market.id] = task
                        last_discovery = now
                    except Exception:
                        pass

                # Render dashboard
                if now - last_refresh >= 5:
                    clear_output(wait=True)
                    html_content = render_dashboard(engine, btc_state, elapsed, cycle)
                    display(HTML(html_content))
                    last_refresh = now

                await asyncio.sleep(1)

        except KeyboardInterrupt:
            print("\n⏹ Stopped by user")

        # Cancel all market tasks
        for task in active_tasks.values():
            task.cancel()
        if active_tasks:
            await asyncio.gather(*active_tasks.values(), return_exceptions=True)

        # Final dashboard
        clear_output(wait=True)
        html_content = render_dashboard(engine, btc_state, time.monotonic() - start_time, cycle)
        display(HTML(html_content))

        # Text summary
        total_pnl = engine.get_total_pnl()
        total_fills = sum(len(s.fills) for s in engine.states.values())
        pnl_sign = "+" if total_pnl >= 0 else ""
        print(f"\n{'='*50}")
        print(f"🏁 FINAL: P&L={pnl_sign}${total_pnl:.4f} | Fills={total_fills} | Markets={len(engine.states)}")
        print(f"{'='*50}")
        engine.print_summary()

# Run it!
await run_paper_mm_dashboard(
    gamma=GAMMA,
    order_size_usdc=ORDER_SIZE,
    requote_interval=REQUOTE_SEC,
    max_markets=MAX_MARKETS,
    duration_sec=600,  # 10 minutes — change as needed
)